# Reflection and Human-in-the-Loop

**What you will build:** two patterns that make workflows more reliable — **reflection** (the model checks and improves its own output in a loop) and **human-in-the-loop** (the workflow pauses for a person to approve an action). These map to chapters 05 and 04d of the n8n course.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/05_reflection_and_hitl.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1. Run once.
!pip install -q openai pydantic

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

def ask(system, user, temperature=0):
    """One-shot helper: system + user in, text out."""
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    return r.choices[0].message.content

print("Ready. Model:", MODEL)

## Reflection: generate, critique, improve

A single generation is one shot in the dark. Reflection adds a loop: produce a draft, check it against the requirements, and revise until it passes (or you hit a limit).

```
  generate ─▶ check ─▶ pass?  ── yes ─▶ done
     ▲                   │
     └──── revise ◀──── no
```

The main design choice is **who does the checking**. A hierarchy, cheapest and most reliable first:

| Check type | Example | Use when |
|------------|---------|----------|
| **Code rule** | `len(words) <= 40` | The requirement is objective — always prefer this |
| **LLM judge** | "Is this friendly and on-topic?" | The requirement is subjective |

Prefer a plain Python check whenever the rule is objective — it's free, instant, and never hallucinates. Use an LLM judge only for things code can't measure.

## A reflection loop with a hard, code-checked constraint

Task: write a tagline that is **exactly 6 words** and **contains the word "quiet"**. Both are objective, so the validator is Python, not an LLM — and an exact count plus a required word is hard enough that the model reliably misses on the first try, so you actually see the loop *loop*.

In [ ]:
def word_count(text):
    return len(text.strip().split())

def write_tagline(product, max_tries=5):
    # Two objective constraints: EXACTLY 6 words AND contains 'quiet'.
    def ok(t): return word_count(t) == 6 and "quiet" in t.lower()
    draft = ask("You write punchy product taglines.",
                f"Write a tagline for {product}. It must be EXACTLY 6 words and include the word 'quiet'.")
    for attempt in range(max_tries):
        n, has = word_count(draft), ("quiet" in draft.lower())
        print(f"attempt {attempt+1}: ({n} words, has 'quiet': {has}) {draft!r}")
        if ok(draft):
            return draft                                   # passed BOTH code checks
        draft = ask("You revise taglines to fit exact constraints without losing punch.",
                    f"Product: {product}\nRewrite as EXACTLY 6 words that include the word 'quiet'.\n"
                    f"Current ({n} words, has 'quiet': {has}): {draft}")
    return draft

print("\nFINAL:", write_tagline("a noise-cancelling coffee mug that keeps drinks hot"))

## When the check is subjective: an LLM judge

Some requirements can't be measured with code ("is this reply empathetic?"). Then a second LLM call acts as a critic. Keep the critic's job narrow and make it output a machine-readable verdict so your loop can branch on it.

In [ ]:
from pydantic import BaseModel

class Verdict(BaseModel):
    passes: bool
    feedback: str

def judge(requirement, text):
    raw = client.chat.completions.create(
        model=MODEL, temperature=0, response_format={"type":"json_object"},
        messages=[{"role":"system","content":
                   f"You are a strict reviewer. Requirement: {requirement}. "
                   f"Reply JSON: {{\"passes\": bool, \"feedback\": str}}."},
                  {"role":"user","content":text}])
    return Verdict.model_validate_json(raw.choices[0].message.content)

# A deliberately handicapped first draft — terse and unapologetic — so you can
# watch the loop actually iterate (same trick as the tagline task):
draft = ask("You write terse, one-sentence support replies. Never apologize.",
            "Reply to: 'Your app deleted my data and I am furious.'")
v = judge("The reply must apologize, stay calm, and offer a concrete next step.", draft)
print("draft:", draft, "\n\nverdict:", v)

### Closing the loop: revise *with the judge's feedback*

The non-obvious part isn't calling the judge — it's what you do with the verdict. The `feedback` goes **back into the revision prompt**. That's what makes iteration converge instead of wandering: the reviser knows exactly what to fix, not just that something failed.

In [ ]:
def improve_until(requirement, first_draft, max_tries=3):
    draft = first_draft
    for attempt in range(1, max_tries + 1):
        v = judge(requirement, draft)
        print(f"attempt {attempt}: passes={v.passes}  feedback={v.feedback!r}")
        if v.passes:
            return draft
        draft = ask(
            "You revise customer support replies. Apply the reviewer's feedback exactly. "
            "Output only the reply.",
            f"Requirement: {requirement}\n"
            f"Reviewer feedback: {v.feedback}\n\n"
            f"Current reply:\n{draft}")
    return draft                       # best effort after max_tries — never loop forever

final = improve_until(
    "The reply must apologize, stay calm, and offer a concrete next step.",
    draft)
print("\nFINAL:", final)

In practice you combine both kinds of check: **code rules first** (free, instant, objective), the **LLM judge only for the subjective residue** — and you pass *both* kinds of feedback to the reviser. Only the check changed between the tagline loop and this one; reflection is always the same loop, and the interesting decision is always *what the check is*.

Reflection has a price worth stating plainly: each round adds **two model calls** (judge + revise), so `max_tries=3` can multiply the cost of one generation by ~5–7×. Pay it when the quality bar matters more than the cost of a retry — customer-facing text, generated code, anything checked once and used many times. Skip it for low-stakes output.

```{note}
Modern **reasoning models** do a form of internal reflection before answering — you're paying for hidden "thinking" tokens that often replace a self-critique round. What they do *not* replace is a check **you control**: your word-count rule, your test suite, your reviewer's rubric. When the requirement is external and checkable, the explicit loop with your own check is still what turns "usually fine" into a guarantee.
```

## Human-in-the-loop: pause before acting

Some actions shouldn't happen without a person's OK — sending an email, spending money, deleting data. The pattern: the workflow *proposes*, then waits for approval before the irreversible step. In a notebook we use `input()`; in production it's a webhook or a UI.

`input()` hides the hard part, so let's name it: a real service **can't block a process for hours** waiting for a human to click. The production shape is *save-and-resume* — persist the pending state (the draft, plus where in the workflow you stopped), return immediately, and resume in a **new** process when the decision arrives, maybe days later. That need to freeze and thaw a running workflow is exactly why checkpointers exist (2.3) and why LangGraph's `interrupt` (2.4) requires one. Keep this in mind and 2.4 will feel inevitable rather than like a framework feature.

In [ ]:
def draft_and_send_email(request):
    draft = ask("You draft short, professional emails. Output only the email body.", request)
    print("----- PROPOSED EMAIL -----")
    print(draft)
    print("--------------------------")

    decision = input("Send this email? (y = send / n = cancel): ").strip().lower()
    if decision == "y":
        # the real 'send' would go here (SMTP, an API call, ...)
        return "Email sent."
    return "Cancelled — nothing was sent."

print(draft_and_send_email("Ask the vendor to reschedule Tuesday's demo to Thursday."))

```{tip}
The approval gate goes **around the irreversible action**, not around the whole workflow. Let the model draft freely; only stop it at the step that touches the outside world. That keeps the human in control of what matters and out of the way of what doesn't.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Combine both checks.** In `write_tagline`, after the word-count passes, add a `judge(...)` call for "is it catchy?" and loop again if it isn't. Code check first, judge second — the cheap gate runs before the expensive one.
2. **Add an edit option.** Give the email approval a third choice, `e` = edit, that asks the user for feedback and regenerates the draft with it.
3. **Budget the loop.** Add a token tally to `improve_until` (as in 0.3: sum `r.usage.total_tokens` — you'll need to return usage from `ask` and `judge`) and print the total when it finishes. **Done when:** you can say out loud what one reflected reply costs versus one unreflected reply.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Combine both checks:**

```python
def write_tagline_v2(product, max_tries=5):
    draft = write_tagline(product)                      # passes the CODE checks first
    for _ in range(max_tries):
        v = judge("The tagline must be catchy and memorable.", draft)
        if v.passes:
            return draft
        draft = ask("You revise taglines to fit exact constraints without losing punch.",
                    f"Keep EXACTLY 6 words including 'quiet'. Reviewer feedback: {v.feedback}\n"
                    f"Current: {draft}")
        if word_count(draft) != 6 or "quiet" not in draft.lower():
            continue                                     # the revision broke a code rule -> next round re-judges anyway
    return draft
```

Note the trap the solution handles: revising for *catchiness* can silently break the *word count*. Every revision must re-pass **all** the checks, not just the one that failed.

**2 — Edit option:**

```python
decision = input("Send this email? (y = send / n = cancel / e = edit): ").strip().lower()
if decision == "e":
    feedback = input("What should change? ")
    draft = ask("You draft short, professional emails. Output only the email body.",
                f"{request}\nRevision instructions from the user: {feedback}")
    # ...then show the new draft and ask again (wrap the whole thing in a while loop)
```

**3 — Budget the loop:** make the helpers return `(text, tokens)` and sum. Typical numbers: a single reply ≈ a few hundred tokens; three reflection rounds ≈ 5–7× that. The absolute cost is tiny here — the *ratio* is what scales to production.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| `ValidationError` from `Verdict` | Provider without JSON mode, or the judge added prose around the JSON | Retry, switch models, or port the retry-on-error `extract()` from 0.2 |
| The judge passes everything | Judges default to politeness | Lower the temperature (already 0 here), make the rubric concrete ("passes only if ALL THREE: apology, calm tone, concrete step"), or ask for a score 1–5 and pass on ≥4 |
| The judge fails everything | Requirement is ambiguous or self-contradictory | Rewrite the requirement as a checklist; test the judge on a reply you know is good |
| The tagline loop never converges | Constraint is genuinely hard for the model, or the reviser keeps trading one rule for another | Keep the `max_tries` cap (that's what it's for); include ALL constraints in every revision prompt |
::::

**What's next:** in **0.6** we finish Block 0 with **memory** — how to carry a conversation across turns without blowing past the context window.